In [1]:

# imports
import os
import sys
import types
import json
import base64

# figure size/format
fig_width = 5.5
fig_height = 3.5
fig_format = 'pdf'
fig_dpi = 300
interactivity = ''
is_shiny = False
is_dashboard = False
plotly_connected = True

# matplotlib defaults / format
try:
  import matplotlib.pyplot as plt
  plt.rcParams['figure.figsize'] = (fig_width, fig_height)
  plt.rcParams['figure.dpi'] = fig_dpi
  plt.rcParams['savefig.dpi'] = "figure"

  # IPython 7.14 deprecated set_matplotlib_formats from IPython
  try:
    from matplotlib_inline.backend_inline import set_matplotlib_formats
  except ImportError:
    # Fall back to deprecated location for older IPython versions
    from IPython.display import set_matplotlib_formats
    
  set_matplotlib_formats(fig_format)
except Exception:
  pass

# plotly use connected mode
try:
  import plotly.io as pio
  if plotly_connected:
    pio.renderers.default = "notebook_connected"
  else:
    pio.renderers.default = "notebook"
  for template in pio.templates.keys():
    pio.templates[template].layout.margin = dict(t=30,r=0,b=0,l=0)
except Exception:
  pass

# disable itables paging for dashboards
if is_dashboard:
  try:
    from itables import options
    options.dom = 'fiBrtlp'
    options.maxBytes = 1024 * 1024
    options.language = dict(info = "Showing _TOTAL_ entries")
    options.classes = "display nowrap compact"
    options.paging = False
    options.searching = True
    options.ordering = True
    options.info = True
    options.lengthChange = False
    options.autoWidth = False
    options.responsive = True
    options.keys = True
    options.buttons = []
  except Exception:
    pass
  
  try:
    import altair as alt
    # By default, dashboards will have container sized
    # vega visualizations which allows them to flow reasonably
    theme_sentinel = '_quarto-dashboard-internal'
    def make_theme(name):
        nonTheme = alt.themes._plugins[name]    
        def patch_theme(*args, **kwargs):
            existingTheme = nonTheme()
            if 'height' not in existingTheme:
              existingTheme['height'] = 'container'
            if 'width' not in existingTheme:
              existingTheme['width'] = 'container'

            if 'config' not in existingTheme:
              existingTheme['config'] = dict()
            
            # Configure the default font sizes
            title_font_size = 15
            header_font_size = 13
            axis_font_size = 12
            legend_font_size = 12
            mark_font_size = 12
            tooltip = False

            config = existingTheme['config']

            # The Axis
            if 'axis' not in config:
              config['axis'] = dict()
            axis = config['axis']
            if 'labelFontSize' not in axis:
              axis['labelFontSize'] = axis_font_size
            if 'titleFontSize' not in axis:
              axis['titleFontSize'] = axis_font_size  

            # The legend
            if 'legend' not in config:
              config['legend'] = dict()
            legend = config['legend']
            if 'labelFontSize' not in legend:
              legend['labelFontSize'] = legend_font_size
            if 'titleFontSize' not in legend:
              legend['titleFontSize'] = legend_font_size  

            # The header
            if 'header' not in config:
              config['header'] = dict()
            header = config['header']
            if 'labelFontSize' not in header:
              header['labelFontSize'] = header_font_size
            if 'titleFontSize' not in header:
              header['titleFontSize'] = header_font_size    

            # Title
            if 'title' not in config:
              config['title'] = dict()
            title = config['title']
            if 'fontSize' not in title:
              title['fontSize'] = title_font_size

            # Marks
            if 'mark' not in config:
              config['mark'] = dict()
            mark = config['mark']
            if 'fontSize' not in mark:
              mark['fontSize'] = mark_font_size

            # Mark tooltips
            if tooltip and 'tooltip' not in mark:
              mark['tooltip'] = dict(content="encoding")

            return existingTheme
            
        return patch_theme

    # We can only do this once per session
    if theme_sentinel not in alt.themes.names():
      for name in alt.themes.names():
        alt.themes.register(name, make_theme(name))
      
      # register a sentinel theme so we only do this once
      alt.themes.register(theme_sentinel, make_theme('default'))
      alt.themes.enable('default')

  except Exception:
    pass

# enable pandas latex repr when targeting pdfs
try:
  import pandas as pd
  if fig_format == 'pdf':
    pd.set_option('display.latex.repr', True)
except Exception:
  pass

# interactivity
if interactivity:
  from IPython.core.interactiveshell import InteractiveShell
  InteractiveShell.ast_node_interactivity = interactivity

# NOTE: the kernel_deps code is repeated in the cleanup.py file
# (we can't easily share this code b/c of the way it is run).
# If you edit this code also edit the same code in cleanup.py!

# output kernel dependencies
kernel_deps = dict()
for module in list(sys.modules.values()):
  # Some modules play games with sys.modules (e.g. email/__init__.py
  # in the standard library), and occasionally this can cause strange
  # failures in getattr.  Just ignore anything that's not an ordinary
  # module.
  if not isinstance(module, types.ModuleType):
    continue
  path = getattr(module, "__file__", None)
  if not path:
    continue
  if path.endswith(".pyc") or path.endswith(".pyo"):
    path = path[:-1]
  if not os.path.exists(path):
    continue
  kernel_deps[path] = os.stat(path).st_mtime
print(json.dumps(kernel_deps))

# set run_path if requested
run_path = 'L1VzZXJzL2FkYW0vZGV2L3VuaS80L2Rpc3Nfd3JpdGV1cA=='
if run_path:
  # hex-decode the path
  run_path = base64.b64decode(run_path.encode("utf-8")).decode("utf-8")
  os.chdir(run_path)

# reset state
%reset

# shiny
# Checking for shiny by using False directly because we're after the %reset. We don't want
# to set a variable that stays in global scope.
if False:
  try:
    import htmltools as _htmltools
    import ast as _ast

    _htmltools.html_dependency_render_mode = "json"

    # This decorator will be added to all function definitions
    def _display_if_has_repr_html(x):
      try:
        # IPython 7.14 preferred import
        from IPython.display import display, HTML
      except:
        from IPython.core.display import display, HTML

      if hasattr(x, '_repr_html_'):
        display(HTML(x._repr_html_()))
      return x

    # ideally we would undo the call to ast_transformers.append
    # at the end of this block whenver an error occurs, we do 
    # this for now as it will only be a problem if the user 
    # switches from shiny to not-shiny mode (and even then likely
    # won't matter)
    import builtins
    builtins._display_if_has_repr_html = _display_if_has_repr_html

    class _FunctionDefReprHtml(_ast.NodeTransformer):
      def visit_FunctionDef(self, node):
        node.decorator_list.insert(
          0,
          _ast.Name(id="_display_if_has_repr_html", ctx=_ast.Load())
        )
        return node

      def visit_AsyncFunctionDef(self, node):
        node.decorator_list.insert(
          0,
          _ast.Name(id="_display_if_has_repr_html", ctx=_ast.Load())
        )
        return node

    ip = get_ipython()
    ip.ast_transformers.append(_FunctionDefReprHtml())

  except:
    pass

def ojs_define(**kwargs):
  import json
  try:
    # IPython 7.14 preferred import
    from IPython.display import display, HTML
  except:
    from IPython.core.display import display, HTML

  # do some minor magic for convenience when handling pandas
  # dataframes
  def convert(v):
    try:
      import pandas as pd
    except ModuleNotFoundError: # don't do the magic when pandas is not available
      return v
    if type(v) == pd.Series:
      v = pd.DataFrame(v)
    if type(v) == pd.DataFrame:
      j = json.loads(v.T.to_json(orient='split'))
      return dict((k,v) for (k,v) in zip(j["index"], j["data"]))
    else:
      return v

  v = dict(contents=list(dict(name=key, value=convert(value)) for (key, value) in kwargs.items()))
  display(HTML('<script type="ojs-define">' + json.dumps(v) + '</script>'), metadata=dict(ojs_define = True))
globals()["ojs_define"] = ojs_define
globals()["__spec__"] = None

{"/opt/homebrew/Cellar/python@3.12/3.12.11/Frameworks/Python.framework/Versions/3.12/lib/python3.12/importlib/_bootstrap.py": 1748965307.0, "/opt/homebrew/Cellar/python@3.12/3.12.11/Frameworks/Python.framework/Versions/3.12/lib/python3.12/importlib/_bootstrap_external.py": 1748965307.0, "/opt/homebrew/Cellar/python@3.12/3.12.11/Frameworks/Python.framework/Versions/3.12/lib/python3.12/zipimport.py": 1748965307.0, "/opt/homebrew/Cellar/python@3.12/3.12.11/Frameworks/Python.framework/Versions/3.12/lib/python3.12/codecs.py": 1748965307.0, "/opt/homebrew/Cellar/python@3.12/3.12.11/Frameworks/Python.framework/Versions/3.12/lib/python3.12/encodings/aliases.py": 1748965307.0, "/opt/homebrew/Cellar/python@3.12/3.12.11/Frameworks/Python.framework/Versions/3.12/lib/python3.12/encodings/__init__.py": 1748965307.0, "/opt/homebrew/Cellar/python@3.12/3.12.11/Frameworks/Python.framework/Versions/3.12/lib/python3.12/encodings/utf_8.py": 1748965307.0, "/opt/homebrew/Cellar/python@3.12/3.12.11/Frameworks

In [2]:
#| label: fig-kernels
#| fig-cap: Prior samples from GPs with the SE kernel (left) and the periodic kernel (right) under varying hyperparameters.
#| echo: false

import torch
import gpytorch
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(2)

X_plot = torch.linspace(0, 5, 300).double()
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

# --- Left: SE/RBF kernel, varying lengthscale ---
lengthscales = [0.1, 0.5, 1.0]
ax = axes[0]
for ell, color in zip(lengthscales, colors):
    kernel = gpytorch.kernels.ScaleKernel(gpytorch.kernels.RBFKernel()).double()
    kernel.base_kernel.lengthscale = ell
    kernel.outputscale = 1.0
    K = kernel(X_plot).evaluate() + 1e-6 * torch.eye(len(X_plot), dtype=torch.float64)
    prior = gpytorch.distributions.MultivariateNormal(
        torch.zeros_like(X_plot), K
    )
    sample = prior.sample().numpy()
    ax.plot(X_plot.numpy(), sample, color=color, alpha=0.85, linewidth=1.2,
            label=fr'$\ell={ell}$')
ax.set_title('SE kernel')
ax.set_xlabel('$x$')
ax.set_ylabel('$f(x)$')
ax.set_ylim(-3, 3)
ax.legend(framealpha=0.9)
ax.axhline(0, color='grey', linewidth=0.5, linestyle='--')

# --- Right: Periodic kernel, varying period ---
periods = [1.0, 3.0, 5.0]
ax = axes[1]
for p, color in zip(periods, colors):
    kernel = gpytorch.kernels.ScaleKernel(
        gpytorch.kernels.PeriodicKernel()
    ).double()
    kernel.base_kernel.period_length = p
    kernel.base_kernel.lengthscale = 1.0
    kernel.outputscale = 1.0
    K = kernel(X_plot).evaluate() + 1e-6 * torch.eye(len(X_plot), dtype=torch.float64)
    prior = gpytorch.distributions.MultivariateNormal(
        torch.zeros_like(X_plot), K
    )
    sample = prior.sample().numpy()
    ax.plot(X_plot.numpy(), sample, color=color, alpha=0.85, linewidth=1.2,
            label=fr'$p={p}$')
ax.set_title('Periodic kernel')
ax.set_xlabel('$x$')
ax.set_ylabel('$f(x)$')
ax.set_ylim(-3, 3)
ax.legend(framealpha=0.9)
ax.axhline(0, color='grey', linewidth=0.5, linestyle='--')

fig.tight_layout()
plt.show()

<Figure size 3300x1200 with 2 Axes>

In [3]:
#| label: fig-gp-regression
#| fig-cap: 'GP regression with an SE kernel. Left: five prior samples before observing data. Right: posterior mean (blue), 95% credible interval (shaded), and five posterior samples after conditioning on five observations (red dots).'
#| echo: false

import torch
import gpytorch
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(3)

# --- GPyTorch model for exact GP regression ---
class ExactGPModel(gpytorch.models.ExactGP):
    def __init__(self, train_x, train_y, likelihood):
        super().__init__(train_x, train_y, likelihood)
        self.mean_module = gpytorch.means.ZeroMean()
        self.covar_module = gpytorch.kernels.ScaleKernel(
            gpytorch.kernels.RBFKernel()
        )

    def forward(self, x):
        return gpytorch.distributions.MultivariateNormal(
            self.mean_module(x), self.covar_module(x)
        )

# --- Setup ---
n_grid = 200
X_plot_np = np.linspace(-5, 5, n_grid)
X_plot = torch.tensor(X_plot_np, dtype=torch.float64)

# Build prior kernel and sample
se_kernel = gpytorch.kernels.ScaleKernel(gpytorch.kernels.RBFKernel())
se_kernel.base_kernel.lengthscale = 1.5
se_kernel.outputscale = 1
se_kernel = se_kernel.double()
K_prior = se_kernel(X_plot).evaluate()
K_prior = K_prior + 1e-6 * torch.eye(n_grid, dtype=torch.float64)
prior_dist = gpytorch.distributions.MultivariateNormal(
    torch.zeros(n_grid, dtype=torch.float64),
    K_prior
)

fig, axes = plt.subplots(1, 2, figsize=(10, 3.8), sharey=True)
BAND_COLOR  = '#1f77b4'
MEAN_COLOR  = '#1f77b4'
BAND_ALPHA  = 0.07
SAMPLE_LW   = 1.0
SAMPLE_ALPHA = 1

# Left: prior samples
ax = axes[0]
prior_mean = np.zeros(n_grid)
prior_std = K_prior.diagonal().sqrt().detach().numpy()
prior_samples = prior_dist.sample(torch.Size([5]))
ax.fill_between(X_plot_np, prior_mean - 2 * prior_std, prior_mean + 2 * prior_std,
                alpha=BAND_ALPHA, color=BAND_COLOR, label='$\\pm 2\\sigma$')
ax.plot(X_plot_np, prior_mean, color=MEAN_COLOR, linewidth=1.5, linestyle='--', label='Mean')
for i in range(5):
    ax.plot(X_plot_np, prior_samples[i].numpy(), linewidth=SAMPLE_LW, alpha=SAMPLE_ALPHA)
ax.axhline(0, color='grey', linewidth=0.4, linestyle=':')
ax.set_title('Prior $f \\sim \\mathcal{GP}(0, k_{\\mathrm{SE}})$')
ax.set_xlabel('$x$')
ax.set_ylabel('$f(x)$')
ax.set_ylim(-3.5, 3.5)
ax.legend(fontsize=9, framealpha=0.9, loc='upper left')

# Right: posterior via GPyTorch exact inference
ax = axes[1]
rng = np.random.default_rng(12)
X_train_np = np.array([-3.5, -1.5, -1.6, -1.4, 0.0, 2.0, 4.0])
y_train_np = np.sin(X_train_np)
X_train = torch.tensor(X_train_np, dtype=torch.float64)
y_train = torch.tensor(y_train_np, dtype=torch.float64)

likelihood = gpytorch.likelihoods.GaussianLikelihood()
likelihood.noise = 0.04
model = ExactGPModel(X_train, y_train, likelihood).double()
model.covar_module.base_kernel.lengthscale = 1.5
model.covar_module.outputscale = 1.0

# Posterior predictions
model.eval()
likelihood.eval()
with torch.no_grad(), gpytorch.settings.fast_pred_var():
    posterior = model(X_plot)
    mu_post = posterior.mean.numpy()
    std_post = posterior.stddev.numpy()

ax.fill_between(X_plot_np, mu_post - 2 * std_post, mu_post + 2 * std_post,
                alpha=BAND_ALPHA, color=BAND_COLOR, label='$\\pm 2\\sigma$')
ax.plot(X_plot_np, mu_post, color=MEAN_COLOR, linewidth=1.5, linestyle='--', label='Mean')

# Posterior samples
with torch.no_grad():
    post_samples = posterior.sample(torch.Size([5]))
for i in range(5):
    ax.plot(X_plot_np, post_samples[i].numpy(), linewidth=SAMPLE_LW, alpha=SAMPLE_ALPHA)

ax.scatter(X_train_np, y_train_np, c='red', s=40, zorder=5, edgecolors='k',
           linewidths=0.5, label='Observations')
ax.set_title('Posterior $f \\mid \\mathbf{X}, \\mathbf{y}$')
ax.set_xlabel('$x$')
ax.set_ylim(-3.5, 3.5)
ax.legend(fontsize=9, framealpha=0.9, loc='upper left')

fig.tight_layout()
plt.show()

/Users/adam/dev/uni/4/Dissertation/code/.venv/lib/python3.12/site-packages/linear_operator/utils/cholesky.py:41: NumericalWarning: A not p.d., added jitter of 1.0e-08 to the diagonal
  warnings.warn(


<Figure size 3000x1140 with 2 Axes>

In [4]:
#| label: fig-gp-training
#| fig-cap: 'Effect of hyperparameter training on GP fit. Left: GP with a intentionally large initial length-scale and a near-zero noise variance, giving a flat oversmoothed posterior. Centre: negative log marginal likelihood during 50 Adam training iterations. Right: GP after training. The learnt hyperparameter values are shown above the plots.'
#| echo: false

import torch
import gpytorch
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)

X_tr = torch.linspace(-5, 5, 20).double()
y_tr = torch.sin(X_tr) + 0.2 * torch.randn(20, dtype=torch.float64)

X_plt_np = np.linspace(-6, 6, 300)
X_plt = torch.tensor(X_plt_np, dtype=torch.float64)

class GPTrain(gpytorch.models.ExactGP):
    def __init__(self, tx, ty, lik):
        super().__init__(tx, ty, lik)
        self.mean_module = gpytorch.means.ZeroMean()
        self.covar_module = gpytorch.kernels.ScaleKernel(gpytorch.kernels.RBFKernel())
    def forward(self, x):
        return gpytorch.distributions.MultivariateNormal(
            self.mean_module(x), self.covar_module(x))

def gp_predict(model, lik, X):
    model.eval(); lik.eval()
    with torch.no_grad():
        p = model(X)
    return p.mean.numpy(), p.stddev.numpy()

def plot_fit(ax, X_tr, y_tr, xp, mu, sd, title):
    ax.fill_between(xp, mu - 2*sd, mu + 2*sd, alpha=0.12, color='#1f77b4')
    ax.plot(xp, mu, color='#1f77b4', linewidth=1.5, label='Posterior mean')
    ax.plot(xp, np.sin(xp), color='grey', linewidth=1.0, linestyle='--',
            alpha=0.7, label='True $f$')
    ax.scatter(X_tr.numpy(), y_tr.numpy(), c='red', s=25, zorder=5,
               edgecolors='k', linewidths=0.5, label='Data')
    ax.set_ylim(-3, 3); ax.set_xlabel('$x$'); ax.set_title(title)

# Before training
lik0 = gpytorch.likelihoods.GaussianLikelihood().double()
m0 = GPTrain(X_tr, y_tr, lik0).double()
m0.covar_module.base_kernel.lengthscale = 15.0
m0.covar_module.outputscale = 1.0
lik0.noise = 0.001
mu0, sd0 = gp_predict(m0, lik0, X_plt)

# Training via Adam
lik1 = gpytorch.likelihoods.GaussianLikelihood().double()
m1 = GPTrain(X_tr, y_tr, lik1).double()
m1.train(); lik1.train()
optimizer = torch.optim.Adam([*m1.parameters(), *lik1.parameters()], lr=0.1)
mll = gpytorch.mlls.ExactMarginalLogLikelihood(lik1, m1)
nlml_trace = []
for _ in range(50):
    optimizer.zero_grad()
    loss = -mll(m1(X_tr), y_tr)
    loss.backward()
    optimizer.step()
    nlml_trace.append(loss.item())

mu1, sd1 = gp_predict(m1, lik1, X_plt)
ell = m1.covar_module.base_kernel.lengthscale.item()
sn2 = lik1.noise.item()

fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))
plot_fit(axes[0], X_tr, y_tr, X_plt_np, mu0, sd0,
         'Before training\n($\\ell=15.0,\\;\\sigma_n^2=0.001$)')
axes[0].set_ylabel('$f(x)$')

axes[1].plot(nlml_trace, color='#1f77b4', linewidth=1.2)
axes[1].set_xlabel('Iteration')
axes[1].set_ylabel('Negative LML')
axes[1].set_title('Training curve')

plot_fit(axes[2], X_tr, y_tr, X_plt_np, mu1, sd1,
         f'After training\n($\\ell={ell:.2f},\\;\\sigma_n^2={sn2:.3f}$)')
axes[2].legend(fontsize=8, loc='upper right')
fig.tight_layout()
plt.show()

/Users/adam/dev/uni/4/Dissertation/code/.venv/lib/python3.12/site-packages/torch/_compile.py:54: UserWarning: optimizer contains a parameter group with duplicate parameters; in future, this will cause an error; see github.com/pytorch/pytorch/issues/40967 for more information
  return disable_fn(*args, **kwargs)


<Figure size 3900x1050 with 3 Axes>

In [5]:
#| label: fig-dgp-composition
#| fig-cap: 'Compositional structure of a two-layer DGP. Left: the first-layer function $f^{(1)}$ warps the input. Centre: the second-layer function $f^{(2)}$ maps the warped inputs to outputs. Right: the composed function $f^{(2)} \circ f^{(1)}$ exhibits non-stationary behaviour — smooth in some regions and rapidly varying in others — that neither layer alone could produce.'
#| echo: false

import torch
import gpytorch
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(21)

n = 300
x_np = np.linspace(-3, 3, n)

# Layer 1: a smooth nonlinear warp
h_np = x_np + 0.7 * np.tanh(1.8 * x_np) + 0.3 * np.sin(2 * x_np)

# Layer 2: draw a GP sample on the warped inputs using GPyTorch
h_t = torch.tensor(h_np, dtype=torch.float64)
kernel = gpytorch.kernels.ScaleKernel(gpytorch.kernels.RBFKernel()).double()
kernel.base_kernel.lengthscale = 0.8
kernel.outputscale = 1.0
prior = gpytorch.distributions.MultivariateNormal(
    torch.zeros(n, dtype=torch.float64),
    kernel(h_t).evaluate() + 1e-6 * torch.eye(n, dtype=torch.float64)
)
f2 = prior.sample().numpy()

fig, axes = plt.subplots(1, 3, figsize=(12, 3.5))

ax = axes[0]
ax.plot(x_np, h_np, color='#1f77b4', linewidth=1.5)
ax.plot(x_np, x_np, color='grey', linewidth=0.7, linestyle='--', alpha=0.5, label='Identity')
ax.set_xlabel('$x$')
ax.set_ylabel('$f^{(1)}(x)$')
ax.set_title('Layer 1: input warp')
ax.legend(fontsize=8, framealpha=0.9)

ax = axes[1]
ax.plot(h_np, f2, color='#ff7f0e', linewidth=1.5)
ax.set_xlabel('$h = f^{(1)}(x)$')
ax.set_ylabel('$f^{(2)}(h)$')
ax.set_title('Layer 2: GP on warped inputs')

ax = axes[2]
ax.plot(x_np, f2, color='#2ca02c', linewidth=1.5)
ax.set_xlabel('$x$')
ax.set_ylabel('$f^{(2)}(f^{(1)}(x))$')
ax.set_title('Composed function')

for ax in axes:
    ax.axhline(0, color='grey', linewidth=0.4, linestyle='--')

fig.tight_layout()
plt.show()

<Figure size 3600x1050 with 3 Axes>

In [6]:
#| label: fig-tdgp-vs-dgp
#| fig-cap: Illustration of standard DGP versus TDGP layer structure. The standard DGP (left) warps the input space arbitrarily, losing interpretability. The TDGP (right) applies a locally linear rescaling of the original inputs, preserving the connection to the original features.
#| echo: false

import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(42)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Original grid
x = np.linspace(-2, 2, 12)
y = np.linspace(-2, 2, 12)
X, Y = np.meshgrid(x, y)

# --- Left: Standard DGP warp (nonlinear) ---
ax = axes[0]
# A nonlinear warp that distorts the grid
Xw = X + 0.4 * np.sin(1.5 * Y) + 0.2 * np.cos(2 * X)
Yw = Y + 0.3 * np.cos(1.5 * X) + 0.3 * np.sin(2 * Y)

for i in range(len(x)):
    ax.plot(Xw[:, i], Yw[:, i], color='#1f77b4', linewidth=0.7, alpha=0.7)
for j in range(len(y)):
    ax.plot(Xw[j, :], Yw[j, :], color='#1f77b4', linewidth=0.7, alpha=0.7)
ax.set_title('Standard DGP warp', fontsize=11)
ax.set_xlabel('$x_1$')
ax.set_ylabel('$x_2$')
ax.set_xlim(-3, 3)
ax.set_ylim(-3, 3)
ax.set_aspect('equal')

# --- Right: TDGP locally linear rescaling ---
ax = axes[1]
# Locally linear: multiply each coordinate by a smooth, positive function
g1 = 0.5 + 0.5 * np.exp(-0.3 * X**2)  # smooth scaling for dim 1
g2 = 0.8 + 0.3 * np.exp(-0.5 * Y**2)  # smooth scaling for dim 2
Xt = X * g1
Yt = Y * g2

for i in range(len(x)):
    ax.plot(Xt[:, i], Yt[:, i], color='#ff7f0e', linewidth=0.7, alpha=0.7)
for j in range(len(y)):
    ax.plot(Xt[j, :], Yt[j, :], color='#ff7f0e', linewidth=0.7, alpha=0.7)
ax.set_title('TDGP locally linear rescaling', fontsize=11)
ax.set_xlabel('$x_1$')
ax.set_ylabel('$x_2$')
ax.set_xlim(-3, 3)
ax.set_ylim(-3, 3)
ax.set_aspect('equal')

fig.tight_layout()
plt.show()

<Figure size 3000x1200 with 2 Axes>